# Notebook 1: Data Exploration

Initial exploration of the SF Police Incident Reports dataset.  
Uses **pandas** locally — no Spark required.

Dataset: `data/sf_incidents.csv` (796K rows, 35 columns)

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/sf_incidents.csv"

# Load a sample first to inspect schema quickly
sample = pd.read_csv(DATA_PATH, nrows=5000, low_memory=False)
print(f"Sample shape: {sample.shape}")
sample.head(3)

In [ ]:
# Column overview
print("Columns:")
for i, col in enumerate(sample.columns, 1):
    print(f"  {i:2}. {col}")

In [ ]:
# Load full dataset
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Full dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
# Data types and null counts
info = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "unique": df.nunique()
})
info

In [ ]:
# Parse datetime column
df["Incident Datetime"] = pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %I:%M:%S %p")
df["hour"] = df["Incident Datetime"].dt.hour
df["month"] = df["Incident Datetime"].dt.month
df["year"] = df["Incident Datetime"].dt.year

print("Year range:", df["year"].min(), "–", df["year"].max())
print("Date range:", df["Incident Datetime"].min(), "→", df["Incident Datetime"].max())

In [ ]:
# Target variable distribution
cat_counts = df["Incident Category"].value_counts()
print(f"Unique categories: {cat_counts.shape[0]}")
print("\nTop 15:")
cat_counts.head(15)

In [ ]:
# Police district distribution
df["Police District"].value_counts(dropna=False)

In [ ]:
# Geo bounds — sanity check
geo = df[["Latitude", "Longitude"]].dropna()
print(f"Latitude:  {geo['Latitude'].min():.4f} – {geo['Latitude'].max():.4f}")
print(f"Longitude: {geo['Longitude'].min():.4f} – {geo['Longitude'].max():.4f}")
print(f"Rows with coordinates: {len(geo):,} / {len(df):,} ({len(geo)/len(df)*100:.1f}%)")

In [ ]:
# Resolution values
df["Resolution"].value_counts(dropna=False)

In [ ]:
# Records per year
df.groupby("year").size().rename("incident_count")

In [ ]:
# Missing values in ML features
ml_cols = ["Incident Datetime", "Latitude", "Longitude",
           "Police District", "Incident Category"]
print("Missing values in ML feature columns:")
df[ml_cols].isnull().sum()